# TensorDescriptor Template Analysis

**Focus**: Deep dive into `ck::TensorDescriptor` template instantiation costs.

This notebook analyzes:
1. How many times `TensorDescriptor` is instantiated
2. Total time spent instantiating this template
3. Most expensive `TensorDescriptor` instantiations
4. Which files spend the most time on `TensorDescriptor`
5. Common template parameter patterns

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

notebook_dir = Path.cwd()
utils_path = (
    notebook_dir.parent / "utils"
    if notebook_dir.name == "notebooks"
    else notebook_dir / "script" / "build_analysis" / "utils"
)
sys.path.insert(0, str(utils_path))

from trace_parser import (
    iter_trace_files,
    stream_events,
    get_template_events,
    extract_template_detail,
    microseconds_to_seconds,
    microseconds_to_milliseconds,
)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 8)

print("✓ Imports successful")

## 1. Scan for TensorDescriptor Instantiations

In [ ]:
TRACE_DIR = Path.cwd().parent.parent.parent / "build-trace"
trace_files = list(iter_trace_files(TRACE_DIR))

# Collect all TensorDescriptor instantiations
tensor_desc_data = []
file_tensor_stats = defaultdict(lambda: {"count": 0, "total_duration": 0})

print("Scanning for ck::TensorDescriptor instantiations...")
print("(Processing 200 files for detailed analysis)\n")

for trace_file in tqdm(trace_files[:200], desc="Analyzing files"):
    try:
        file_count = 0
        file_duration = 0

        for event in get_template_events(stream_events(trace_file)):
            detail = extract_template_detail(event)
            if detail and "TensorDescriptor" in detail:
                duration = event.get("dur", 0)

                tensor_desc_data.append(
                    {
                        "file": trace_file.name,
                        "template": detail,
                        "event_type": event.get("name"),
                        "duration_ms": microseconds_to_milliseconds(duration),
                        "duration_us": duration,
                    }
                )

                file_count += 1
                file_duration += duration

        if file_count > 0:
            file_tensor_stats[trace_file.name]["count"] = file_count
            file_tensor_stats[trace_file.name]["total_duration"] = file_duration

    except Exception as e:
        print(f"Error processing {trace_file.name}: {e}")

print(f"\n✓ Found {len(tensor_desc_data):,} TensorDescriptor instantiations")
print(f"✓ Across {len(file_tensor_stats)} files")

## 2. Overall TensorDescriptor Statistics

In [ ]:
df = pd.DataFrame(tensor_desc_data)

total_time_sec = df["duration_us"].sum() / 1_000_000
total_time_hours = total_time_sec / 3600

print("=" * 70)
print("TENSORDESCRIPTOR ANALYSIS (200 file sample)")
print("=" * 70)
print(f"Total instantiations:      {len(df):,}")
print(
    f"Total time:                {total_time_hours:.2f} hours ({total_time_sec / 60:.2f} minutes)"
)
print(f"Average per instantiation: {df['duration_ms'].mean():.2f} ms")
print(f"Median per instantiation:  {df['duration_ms'].median():.2f} ms")
print(f"Max single instantiation:  {df['duration_ms'].max():.2f} ms")
print("=" * 70)

# Extrapolate to full build
scale_factor = len(trace_files) / 200
print(f"\nExtrapolated to full build ({len(trace_files)} files):")
print(
    f"  Estimated TensorDescriptor time: ~{total_time_hours * scale_factor:.1f} hours"
)
print(f"  Estimated instantiations: ~{len(df) * scale_factor:,.0f}")

## 3. Files with Most TensorDescriptor Overhead

In [ ]:
# Aggregate by file
file_df = pd.DataFrame(
    [
        {
            "file": filename,
            "count": stats["count"],
            "total_seconds": microseconds_to_seconds(stats["total_duration"]),
            "avg_ms": stats["total_duration"] / stats["count"] / 1000,
        }
        for filename, stats in file_tensor_stats.items()
    ]
).sort_values("total_seconds", ascending=False)

print("Top 20 files by TensorDescriptor compilation time:\n")
for i, row in enumerate(file_df.head(20).itertuples(), 1):
    print(
        f"{i:2d}. {row.total_seconds:7.2f}s  {row.count:4,} instantiations  {row.file}"
    )

## 4. Most Expensive TensorDescriptor Variants

In [ ]:
# Group by template signature (first 150 chars to group similar templates)
template_groups = defaultdict(
    lambda: {"count": 0, "total_duration": 0, "max_duration": 0}
)

for _, row in df.iterrows():
    # Extract template signature (truncate for grouping)
    template = row["template"]
    key = template[:150] if len(template) > 150 else template

    template_groups[key]["count"] += 1
    template_groups[key]["total_duration"] += row["duration_us"]
    template_groups[key]["max_duration"] = max(
        template_groups[key]["max_duration"], row["duration_ms"]
    )

# Convert to DataFrame
template_df = pd.DataFrame(
    [
        {
            "template": name,
            "count": stats["count"],
            "total_seconds": microseconds_to_seconds(stats["total_duration"]),
            "avg_ms": stats["total_duration"] / stats["count"] / 1000,
            "max_ms": stats["max_duration"],
        }
        for name, stats in template_groups.items()
    ]
).sort_values("total_seconds", ascending=False)

print("Top 20 TensorDescriptor variants by total time:\n")
for i, row in enumerate(template_df.head(20).itertuples(), 1):
    template_name = (
        row.template[:100] + "..." if len(row.template) > 100 else row.template
    )
    print(
        f"{i:2d}. {row.total_seconds:6.2f}s  {row.count:4,}x  avg:{row.avg_ms:6.2f}ms  {template_name}"
    )

## 5. Slowest Individual Instantiations

In [ ]:
slowest = df.nlargest(20, "duration_ms")

print("Top 20 slowest individual TensorDescriptor instantiations:\n")
for i, row in enumerate(slowest.itertuples(), 1):
    template_name = (
        row.template[:100] + "..." if len(row.template) > 100 else row.template
    )
    print(f"{i:2d}. {row.duration_ms:7.2f}ms  [{row.event_type}]  {template_name}")
    print(f"    File: {row.file}\n")

## 6. Template Parameter Analysis

Extract common patterns in TensorDescriptor template parameters.

In [ ]:
# Extract template parameter patterns
print("Analyzing template parameter patterns...\n")

# Count Tuple usage
tuple_count = sum(1 for t in df["template"] if "Tuple<" in t)
print(
    f"Templates with Tuple:      {tuple_count:,} ({tuple_count / len(df) * 100:.1f}%)"
)

# Count Embed usage
embed_count = sum(1 for t in df["template"] if "Embed<" in t)
print(
    f"Templates with Embed:      {embed_count:,} ({embed_count / len(df) * 100:.1f}%)"
)

# Count UnMerge usage
unmerge_count = sum(1 for t in df["template"] if "UnMerge<" in t)
print(
    f"Templates with UnMerge:    {unmerge_count:,} ({unmerge_count / len(df) * 100:.1f}%)"
)

# Count Merge usage
merge_count = sum(1 for t in df["template"] if "Merge<" in t)
print(
    f"Templates with Merge:      {merge_count:,} ({merge_count / len(df) * 100:.1f}%)"
)

## 7. Visualization: Distribution of Instantiation Times

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of durations
axes[0].hist(df["duration_ms"], bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Duration (ms)", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)
axes[0].set_title(
    "Distribution of TensorDescriptor Instantiation Times",
    fontsize=12,
    fontweight="bold",
)
axes[0].axvline(
    df["duration_ms"].median(),
    color="red",
    linestyle="--",
    label=f"Median: {df['duration_ms'].median():.2f}ms",
)
axes[0].legend()

# Top files by TensorDescriptor time
top_files = file_df.head(15)
sns.barplot(
    data=top_files,
    x="total_seconds",
    y="file",
    hue="file",
    ax=axes[1],
    palette="rocket",
    legend=False,
)
axes[1].set_title(
    "Top 15 Files by TensorDescriptor Time", fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("Total Duration (seconds)", fontsize=11)
axes[1].set_ylabel("File", fontsize=11)

plt.tight_layout()
plt.show()

## 8. Sample Template Signatures

Show examples of the most common TensorDescriptor patterns.

In [ ]:
print("Sample TensorDescriptor template signatures:\n")
print("=" * 100)

# Show a few examples of different patterns
samples = template_df.head(10)
for i, row in enumerate(samples.itertuples(), 1):
    print(
        f"\n{i}. Count: {row.count:,}  Total: {row.total_seconds:.2f}s  Avg: {row.avg_ms:.2f}ms"
    )
    print(f"   {row.template}")
    print("-" * 100)

## 9. Key Findings & Recommendations

Based on the analysis, we can identify:

### Optimization Opportunities
1. **Frequently instantiated patterns**: Templates instantiated 100+ times are candidates for explicit instantiation
2. **Expensive individual instantiations**: Templates taking >100ms might benefit from simplification
3. **Hot files**: Files spending >10s on TensorDescriptor could be refactored

### Next Steps
- Identify common template parameter combinations for explicit instantiation
- Analyze template depth/nesting levels
- Compare TensorDescriptor overhead across different kernel types
- Profile specific transform operations (Embed, UnMerge, Merge)

## 10. Save Results

In [ ]:
# Save detailed results
output_dir = Path.cwd().parent / "data"

# Save all TensorDescriptor instantiations
df.to_csv(output_dir / "tensor_descriptor_instantiations.csv", index=False)
print(f"✓ Saved {len(df):,} instantiations to tensor_descriptor_instantiations.csv")

# Save aggregated template statistics
template_df.to_csv(output_dir / "tensor_descriptor_templates.csv", index=False)
print(
    f"✓ Saved {len(template_df):,} unique templates to tensor_descriptor_templates.csv"
)

# Save file statistics
file_df.to_csv(output_dir / "tensor_descriptor_by_file.csv", index=False)
print(f"✓ Saved {len(file_df):,} file stats to tensor_descriptor_by_file.csv")